# BlackPearl Morphix — Stage 2, Step 2: Link positions, cell types and knockouts

The Perturb-FISH tables are split up:
- `coordinates.csv` and `allcellsPerturbationTable.csv`: **no header row**, one row per cell, same order, but the 77 knockout columns are **unnamed**.
- `tumorpooledperturbations.csv` (genes × tumour cells) and `perturbed_tcells_perturbation.csv` (genes × T cells): **named genes**, but cells are labelled `Cell_1…` with **no positions**.
- `merfishcounttable.csv`: gene counts for **every** cell, presumably in the same order as the coordinates.

**The trick:** `tumorMerfish.csv` / `perturbed_tcells_merfish.csv` hold the *same count rows* for the perturbed cells. Matching identical rows to `merfishcounttable.csv` tells us where each perturbed cell is.

**Output:** `morphix_pfish_cells.csv.gz`, one row per cell with `x, y, cell_class, gene`. That is the input to the Stage 2 model.

CPU runtime is fine. Downloads ~270 MB; takes ~5 minutes. At the end, copy the **SUMMARY** block and paste it back.

In [ ]:
import os, json, urllib.request
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

BASE = 'https://download.brainimagelibrary.org/0c/bd/0cbd479c521afff9/extras/tumors/processed/finaltables/'
os.makedirs('pfish', exist_ok=True)
def fetch(name):
    path = os.path.join('pfish', name)
    if not os.path.exists(path):
        print('downloading', name, '...')
        req = urllib.request.Request(BASE + name, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=900) as r, open(path, 'wb') as f:
            while True:
                chunk = r.read(1 << 22)
                if not chunk: break
                f.write(chunk)
    return path

for f in ['coordinates.csv', 'allcellsPerturbationTable.csv', 'merfishcounttable.csv', 'tumorMerfish.csv',
          'tumorpooledperturbations.csv', 'perturbed_tcells_merfish.csv', 'perturbed_tcells_perturbation.csv']:
    fetch(f)
S = {}   # summary for pasting back

def head_lines(path, n=3, width=300):
    with open(path) as fh:
        return [fh.readline()[:width].rstrip() for _ in range(n)]
for f in ['coordinates.csv', 'allcellsPerturbationTable.csv', 'merfishcounttable.csv']:
    print(f'--- {f} (raw first lines) ---'); [print(l) for l in head_lines(os.path.join('pfish', f))]

## 1. Load the headerless tables correctly

In [ ]:
def has_header(path):
    first = head_lines(path, 1, 100000)[0].split(',')
    def isnum(t):
        try: float(t); return True
        except ValueError: return t.strip() == ''
    return not all(isnum(t) for t in first[1:6])   # gene names -> header

coords = pd.read_csv('pfish/coordinates.csv', header=0 if has_header('pfish/coordinates.csv') else None)
if coords.shape[1] > 2:          # drop an index column if present
    coords = coords.iloc[:, -2:]
coords.columns = ['x', 'y']
pert_all = pd.read_csv('pfish/allcellsPerturbationTable.csv',
                       header=0 if has_header('pfish/allcellsPerturbationTable.csv') else None)
counts_hdr = has_header('pfish/merfishcounttable.csv')
counts = pd.read_csv('pfish/merfishcounttable.csv', header=0 if counts_hdr else None, dtype=np.float32, low_memory=False) \
    if not counts_hdr else pd.read_csv('pfish/merfishcounttable.csv', header=0, low_memory=False)
if counts_hdr and (str(counts.columns[0]).startswith('Unnamed') or counts.columns[0] == ''):
    counts = counts.set_index(counts.columns[0])
counts = counts.apply(pd.to_numeric, errors='coerce').fillna(0).astype(np.float32)

def drop_index_col(df):
    # a headerless file may still start with a 0,1,2,... row-number column
    first = df.iloc[:, 0].values
    if df.shape[1] > 2 and np.array_equal(first, np.arange(len(df))):
        return df.iloc[:, 1:]
    return df
counts = drop_index_col(counts)
pert_all = drop_index_col(pert_all)

print('coords', coords.shape, '| perturbation table', pert_all.shape, '| count table', counts.shape,
      '| count table has gene header:', counts_hdr)
S['n_coords'], S['n_pert_rows'], S['count_shape'], S['count_header'] = len(coords), len(pert_all), list(counts.shape), counts_hdr
S['rows_align'] = len(coords) == len(pert_all) == len(counts)
print('row counts agree:', S['rows_align'])

## 2. Find each perturbed tumour cell and T cell in the full table

In [ ]:
def load_sub(path):
    df = pd.read_csv(path, low_memory=False)
    if str(df.columns[0]).startswith('Unnamed'):
        df = df.set_index(df.columns[0])
    return df.apply(pd.to_numeric, errors='coerce').fillna(0).astype(np.float32)

def row_keys(arr):
    arr = np.ascontiguousarray(np.round(arr).astype(np.int32))
    return [r.tobytes() for r in arr]

def match_rows(sub, full):
    # align columns: by gene name if both have names, else by position
    if counts_hdr and set(sub.columns) <= set(full.columns):
        cols = list(sub.columns); how = 'by gene name'
    elif sub.shape[1] == full.shape[1]:
        cols = None; how = 'by column order'
    else:
        return None, f'cannot align columns (sub {sub.shape[1]} vs full {full.shape[1]}, no names)'
    F = full[cols].values if cols is not None else full.values
    Sx = sub.values
    keys_full = row_keys(F)
    lookup = {}
    for i, k in enumerate(keys_full):
        lookup.setdefault(k, []).append(i)
    idx = np.full(len(Sx), -1); n_ambig = 0
    for j, k in enumerate(row_keys(Sx)):
        hits = lookup.get(k, [])
        if len(hits) == 1: idx[j] = hits[0]
        elif len(hits) > 1: n_ambig += 1
    return idx, f'{how}: matched {np.sum(idx >= 0)}/{len(Sx)} uniquely, {n_ambig} ambiguous (identical count rows)'

tumor_counts = load_sub('pfish/tumorMerfish.csv')
tcell_counts = load_sub('pfish/perturbed_tcells_merfish.csv')

def _fingerprints(M, w):
    return np.round(M @ w).astype(np.int64)

def find_columns(sub, full, n_probe=800, W=40):
    # Map each gene column of `sub` to a column of `full` (extra columns may sit anywhere).
    # 1) Find 'anchor' cells: probe rows whose values on a window of W genes appear exactly once in `full`.
    # 2) Using those anchor pairs, match every gene column individually by exact equality.
    rng = np.random.default_rng(0)
    n, m = sub.shape[1], full.shape[1]
    Sv = np.round(sub.values).astype(np.float64)
    F = np.round(full.values).astype(np.float64)
    probe_idx = rng.choice(len(Sv), min(n_probe, len(Sv)), replace=False)
    best = (0, None)
    for s0 in range(0, n - W + 1, max(1, (n - W) // 12)):
        w = rng.integers(1, 1_000_003, size=W).astype(np.float64)
        P = Sv[probe_idx, s0:s0 + W]
        rich = P.sum(1) >= 10                                   # informative probe rows only
        if rich.sum() < 20:
            continue
        tp = _fingerprints(P[rich], w)
        for d in range(0, m - n + 1):
            if s0 + d + W > m:
                break
            fp = _fingerprints(F[:, s0 + d:s0 + d + W], w)
            uniq, first, cnt = np.unique(fp, return_index=True, return_counts=True)
            u1, r1 = uniq[cnt == 1], first[cnt == 1]
            pos = np.clip(np.searchsorted(u1, tp), 0, max(len(u1) - 1, 0))
            hit = (len(u1) > 0) & (u1[pos] == tp)
            if hit.sum() > best[0]:
                pairs = list(zip(probe_idx[np.flatnonzero(rich)[hit]], r1[pos[hit]]))
                best = (int(hit.sum()), pairs)
        if best[0] >= 200:
            break
    if best[1] is None or best[0] < 30:
        return None, 0.0, 'no anchor cells found'
    a_sub = np.array([p[0] for p in best[1]]); a_full = np.array([p[1] for p in best[1]])
    A_s, A_f = Sv[a_sub], F[a_full]                              # anchors x genes
    cols = np.full(n, -1)
    used = set()
    for j in range(n):
        eq = np.flatnonzero(np.all(A_f == A_s[:, [j]], axis=0))
        eq = [k for k in eq if k not in used]
        if eq:
            # tie-break: column nearest the previous mapped column (order is usually preserved)
            prev = cols[j - 1] if j > 0 and cols[j - 1] >= 0 else -1
            k = min(eq, key=lambda k: (k <= prev, abs(k - prev - 1)))
            cols[j] = k; used.add(k)
    ok = (cols >= 0).mean()
    if ok < 1:
        return None, ok, f'{best[0]} anchors, but only {100 * ok:.1f}% of gene columns matched'
    # verify on all rows: fraction of probe rows found exactly with the full mapping
    w = rng.integers(1, 1_000_003, size=n).astype(np.float64)
    sig = set(_fingerprints(F[:, cols], w))
    rich = Sv[probe_idx].sum(1) >= 5
    frac = float(np.mean([t in sig for t in _fingerprints(Sv[probe_idx][rich], w)]))
    contiguous = bool(np.all(np.diff(cols) == 1))
    return cols, frac, f'{best[0]} anchor cells; columns {"contiguous" if contiguous else "non-contiguous"} ' \
                       f'({cols.min()}..{cols.max()})'

if not counts_hdr:
    cols, frac, how = find_columns(tumor_counts, counts)
    print(f'best column alignment: {how} -> {100 * frac:.1f}% of probe tumour rows found exactly')
    S['column_alignment'] = f'{how} ({100 * frac:.1f}% probe rows found)'
    S['extra_columns_not_used'] = [int(i) for i in np.setdiff1d(np.arange(counts.shape[1]), cols)][:60]
    if cols is not None and frac > 0.5:
        counts = counts.iloc[:, cols]
        counts.columns = list(tumor_counts.columns)
        counts_hdr = True
    else:
        print('WARNING: could not find the gene columns; paste the SUMMARY back so we can investigate.')

tumor_idx, msg_t = match_rows(tumor_counts, counts); print('tumour cells:', msg_t)
if tumor_idx is None:
    tumor_idx = np.full(len(tumor_counts), -1)
tcell_idx, msg_c = match_rows(tcell_counts, counts); print('T cells     :', msg_c)
if tcell_idx is None:
    tcell_idx = np.full(len(tcell_counts), -1)
S['tumour_match'], S['tcell_match'] = msg_t, msg_c

## 3. Assign one gene knockout per cell

In [ ]:
def gene_per_cell(path):
    g = pd.read_csv(path, index_col=0)           # genes x cells
    B = (g.apply(pd.to_numeric, errors='coerce').fillna(0).values > 0)
    n = B.sum(0)
    gene = np.where(n == 1, np.array(g.index)[B.argmax(0)], np.where(n == 0, 'none', 'multiple'))
    return gene, list(g.index), g.values.max()

tumor_gene, tumor_genes, tmax = gene_per_cell('pfish/tumorpooledperturbations.csv')
tcell_gene, tcell_genes, cmax = gene_per_cell('pfish/perturbed_tcells_perturbation.csv')
print('tumour knockout genes:', tumor_genes)
print('T-cell knockout genes:', tcell_genes)
print('max values in tables (binary if 1):', tmax, cmax)

cells = coords.copy()
cells['cell_class'] = 'other'
cells['gene'] = pd.Series([''] * len(cells), dtype=object)
ok = tumor_idx >= 0
cells.loc[tumor_idx[ok], 'cell_class'] = 'tumour'
cells.loc[tumor_idx[ok], 'gene'] = tumor_gene[ok]
ok2 = tcell_idx >= 0
cells.loc[tcell_idx[ok2], 'cell_class'] = 'tcell'
cells.loc[tcell_idx[ok2], 'gene'] = tcell_gene[ok2]
cells['total_counts'] = counts.values.sum(1)

vc = cells.loc[cells.cell_class == 'tumour', 'gene'].value_counts()
print('\ntumour cells per knockout:'); print(vc.to_string())
S['tumour_genes'] = tumor_genes; S['tcell_genes'] = tcell_genes
S['tumour_cells_per_gene'] = vc.to_dict()
S['tcells_per_gene'] = cells.loc[cells.cell_class == 'tcell', 'gene'].value_counts().to_dict()
S['class_counts'] = cells.cell_class.value_counts().to_dict()

## 4. Name the 77 unnamed knockout columns (bonus check)

In [ ]:
A = (pert_all.values > 0)
rows = tumor_idx[(tumor_idx >= 0) & np.isin(tumor_gene, tumor_genes)]
genes_for_rows = tumor_gene[(tumor_idx >= 0) & np.isin(tumor_gene, tumor_genes)]
col_names = {}
for c in range(A.shape[1]):
    hit = A[rows, c]
    if hit.sum() == 0: continue
    g, n = np.unique(genes_for_rows[hit], return_counts=True)
    col_names[c] = f'{g[n.argmax()]} ({n.max()}/{hit.sum()})'
print('column -> most common gene among matched tumour cells (agreement):')
for c, v in sorted(col_names.items()): print(f'  col {c:2d}: {v}')
S['column_names'] = col_names

## 5. Look at it and save

In [ ]:
fig, ax = plt.subplots(figsize=(11, 8))
o = cells[cells.cell_class == 'other']
ax.scatter(o.x, o.y, s=0.2, c='lightgrey', label=f'other ({len(o)})')
for cls, col in [('tcell', 'tab:blue'), ('tumour', 'tab:red')]:
    d = cells[cells.cell_class == cls]
    ax.scatter(d.x, d.y, s=1, c=col, label=f'{cls} with knockout ({len(d)})')
ax.set_aspect('equal'); ax.legend(markerscale=8); ax.set_title('Perturbed cells within the tumour sections')
plt.show()

# how far apart are perturbed tumour cells? (are knockouts clustered or scattered?)
from scipy.spatial import cKDTree
tum = cells[(cells.cell_class == 'tumour') & (cells.gene.isin(tumor_genes))]
d_all, _ = cKDTree(cells[['x', 'y']].values).query(cells[['x', 'y']].values, k=2)
d_t, _ = cKDTree(tum[['x', 'y']].values).query(tum[['x', 'y']].values, k=2)
same = []
tree = cKDTree(tum[['x', 'y']].values)
_, nb = tree.query(tum[['x', 'y']].values, k=6)
gv = tum.gene.to_numpy(dtype=object)
same_frac = (gv[nb[:, 1:]] == gv[:, None]).mean()
S['median_nn_all_cells'] = float(np.median(d_all[:, 1]))
S['median_nn_between_perturbed_tumour_cells'] = float(np.median(d_t[:, 1]))
S['frac_5nn_perturbed_neighbours_same_gene'] = float(same_frac)
S['expected_same_gene_if_random'] = float(((tum.gene.value_counts() / len(tum)) ** 2).sum())
print('median NN distance, all cells:', S['median_nn_all_cells'])
print('median distance between perturbed tumour cells:', S['median_nn_between_perturbed_tumour_cells'])
print(f'fraction of nearby perturbed neighbours with the SAME knockout: {same_frac:.3f} '
      f'(random expectation {S["expected_same_gene_if_random"]:.3f}) -> >> random means clonal patches')

cells.to_csv('morphix_pfish_cells.csv.gz', index_label='cell_id')
print('saved morphix_pfish_cells.csv.gz', cells.shape)
try:
    from google.colab import files; files.download('morphix_pfish_cells.csv.gz')
except Exception:
    pass

## SUMMARY — copy everything printed below and paste it back

In [ ]:
print('===== SUMMARY START =====')
print(json.dumps(S, indent=1, default=str)[:15000])
print('===== SUMMARY END =====')